# 6.02 Markdown / HTML / Code / LaTeX Splitter

**구조를 이해하는** 분할기 모음. 헤더 계층 · HTML DOM · 언어별 문법 경계를 존중해 자르므로, metadata 가 자동으로 채워지고 검색 정확도가 크게 올라간다.

## 학습 목표

- `MarkdownHeaderTextSplitter` 로 `#/##/###` 헤더 계층을 metadata 로 받는다
- `HTMLHeaderTextSplitter` · `HTMLSectionSplitter` 로 DOM 태그 기준 분할
- `RecursiveCharacterTextSplitter.from_language(Language.PYTHON)` 로 코드의 **함수·클래스 경계** 분할
- `LatexTextSplitter` 로 섹션/환경 기반 LaTeX 분할
- metadata 가 실린 chunk 를 vector store 에 넣고 **헤더별 필터 검색**

## 언제 쓰나

- **기술 문서 / API 레퍼런스**: Markdown/HTML 헤더 = 의미 경계
- **코드 RAG**: 토큰 단위 분할은 치명적. `from_language` 가 AST 에 가까운 분할
- **논문 / 레포트**: LaTeX 소스면 `LatexTextSplitter` 로 `\section` 경계 보존

## 6.02.1 환경 설정

필요 패키지: `langchain-text-splitters`, `lxml`(HTML splitter 용). 외부 API 키 **불필요**.

In [ ]:
# !pip install -U langchain-text-splitters lxml

from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    HTMLHeaderTextSplitter,
    HTMLSectionSplitter,
    RecursiveCharacterTextSplitter,
    LatexTextSplitter,
    Language,
)
print('import OK')

## 6.02.2 `MarkdownHeaderTextSplitter` — 헤더 기반 분할

`headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")]` 을 주면 헤더가 바뀔 때마다 새 chunk 가 만들어지고 **metadata 에 헤더 계층이 자동 기록** 된다. `strip_headers=True`(기본값) 면 chunk 본문에서 `#` 라인을 제거해 임베딩을 깨끗하게 유지.

실무에서는 header splitter 로 **1단계** 자르고, 큰 chunk 는 `RecursiveCharacterTextSplitter` 로 **2단계** 내린다 (chained splitting). `Document` metadata 는 자동 승계.

In [ ]:
MARKDOWN = '''
# LangChain RAG 가이드

LLM 앱에서 검색 기반 생성을 구현하는 실전 가이드.

## 1. 문서 로더

PDF · HTML · Notion 등 70+ 소스를 지원한다.

### 1.1 PyPDFLoader

페이지 단위로 `Document` 를 반환한다.

### 1.2 WebBaseLoader

`bs4` 기반 HTML 스크래퍼.

## 2. 분할기

이 노트북의 주제.

### 2.1 Recursive

가장 널리 쓰이는 일반 분할기.

### 2.2 Markdown

헤더 계층을 metadata 로 보존.
'''

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")],
    strip_headers=True,
)
md_docs = md_splitter.split_text(MARKDOWN)
print(f'[1단계] header chunks: {len(md_docs)}')
for d in md_docs:
    print(d.metadata, '|', d.page_content[:40].replace('\n', ' '), '…')

chain_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
final_docs = chain_splitter.split_documents(md_docs)
print(f'\n[2단계] after recursive: {len(final_docs)} chunks (metadata 자동 승계)')

## 6.02.3 `HTMLHeaderTextSplitter` · `HTMLSectionSplitter`

HTML DOM 에서 `<h1>..<h6>` 를 기준으로 자른다. Markdown 분할기와 같은 자동 metadata 규칙.

- `HTMLHeaderTextSplitter` — 문자열/URL 입력, 헤더 단위로 분할
- `HTMLSectionSplitter` — xsl 기반으로 `<section>` 까지 확장 인식 (최신 웹 표준에 더 정확)

In [ ]:
HTML = '''
<!doctype html>
<html><body>
<h1>LangChain 문서</h1>
<p>오픈소스 LLM 프레임워크.</p>
<h2>설치</h2>
<p>pip install langchain</p>
<h2>예제</h2>
<h3>첫 예제</h3>
<p>ChatModel 생성.</p>
<h3>RAG 예제</h3>
<p>vector store 연결.</p>
</body></html>
'''

html_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=[("h1", "Title"), ("h2", "Section"), ("h3", "Subsection")],
)
html_docs = html_splitter.split_text(HTML)
print(f'[HTMLHeader ] chunks: {len(html_docs)}')
for d in html_docs:
    print(d.metadata, '|', d.page_content[:40].replace('\n', ' '), '…')

section_splitter = HTMLSectionSplitter(
    headers_to_split_on=[("h1", "Title"), ("h2", "Section")],
)
sec_docs = section_splitter.split_text(HTML)
print(f'\n[HTMLSection] chunks: {len(sec_docs)}')

## 6.02.4 `RecursiveCharacterTextSplitter.from_language(...)` — 30+ 언어

각 언어의 대표 구분자(함수 선언, 클래스 선언, 블록 주석 등) 를 미리 정의해 **문법 경계를 깨지 않고** 재귀 분할한다. 지원 언어 enum 멤버는 아래처럼 직접 확인 가능.

In [ ]:
# Language enum 전체 멤버 — 드리프트 방지를 위해 직접 probe
langs = [m for m in dir(Language) if m.isupper() and not m.startswith('_')]
print(f'지원 언어 {len(langs)}종:')
print(langs)

### Python 코드 분할 + 프리셋 separator 확인

`PYTHON` 프리셋은 `\nclass `, `\ndef `, `\n\tdef ` 등을 우선 separator 로 둔다 → 함수/클래스 경계 보존.
`get_separators_for_language(Language.X)` 로 각 언어의 구분자를 직접 확인할 수 있다.

In [ ]:
PY_CODE = '''
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def search(query: str) -> str:
    """웹 검색 결과를 반환한다."""
    return f"results for {query}"

@tool
def summarize(text: str) -> str:
    """주어진 텍스트를 3문장으로 요약한다."""
    return text[:100]

class Researcher:
    def __init__(self, model):
        self.agent = create_agent(
            model=model,
            tools=[search, summarize],
            system_prompt="You are a research assistant.",
        )

    def run(self, question: str):
        return self.agent.invoke({"messages": [{"role": "user", "content": question}]})
'''

py_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=200,
    chunk_overlap=20,
)
py_chunks = py_splitter.split_text(PY_CODE)
print(f'[PYTHON] chunks: {len(py_chunks)}')
for i, c in enumerate(py_chunks):
    head = (c.splitlines() or [''])[0]
    print(f'  [{i}] {len(c):>3}자 | {head[:50]}')

# 다른 언어들의 separator 프리셋 비교
print()
for lang in [Language.JS, Language.GO, Language.SOL, Language.LATEX]:
    seps = RecursiveCharacterTextSplitter.get_separators_for_language(lang)
    print(f'{lang.name:>8}: {seps[:4]} …  (총 {len(seps)}개)')

## 6.02.5 `LatexTextSplitter` — LaTeX 전용

`RecursiveCharacterTextSplitter` 를 `Language.LATEX` 로 고정한 편의 서브클래스. `\chapter`, `\section`, `\begin{...}` 등 LaTeX 구조를 separator 로 사용.

In [ ]:
TEX = r'''
\section{Introduction}
This paper investigates retrieval-augmented generation.
\subsection{Motivation}
LLMs hallucinate on knowledge-intensive tasks.
\section{Method}
We propose a two-stage retrieval pipeline.
\subsection{Indexing}
Documents are chunked and embedded.
\subsection{Retrieval}
Top-k nearest neighbors are concatenated into the prompt.
'''

tex_splitter = LatexTextSplitter(chunk_size=120, chunk_overlap=10)
tex_chunks = tex_splitter.split_text(TEX)
print(f'latex chunks: {len(tex_chunks)}')
for i, c in enumerate(tex_chunks):
    print(f'  [{i}] {c[:60].strip()!r}')

## 6.02.6 vector store 연동 — 헤더 metadata 로 필터 검색

`MarkdownHeaderTextSplitter` → `InMemoryVectorStore` → `similarity_search(..., filter=...)`. 섹션 단위 검색이 필요한 기술 문서 RAG 의 전형적 패턴.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.embeddings import DeterministicFakeEmbedding

emb = DeterministicFakeEmbedding(size=128)
store = InMemoryVectorStore.from_documents(md_docs, embedding=emb)

# 자유 쿼리
print('--- 자유 검색 ---')
for h in store.similarity_search("PyPDFLoader 사용법", k=3):
    print(h.metadata, '|', h.page_content[:40].replace('\n', ' '), '…')

# H2='2. 분할기' 하위만 필터
print('\n--- H2="2. 분할기" 필터 ---')
for h in store.similarity_search(
    "Markdown 분할",
    k=3,
    filter=lambda d: d.metadata.get("H2") == "2. 분할기",
):
    print(h.metadata, '|', h.page_content[:40].replace('\n', ' '), '…')

## 체크리스트

- [ ] Markdown/HTML 문서는 header splitter → recursive 순으로 체인
- [ ] 코드 RAG 에는 `from_language(Language.X)` 를 기본 선택
- [ ] `Language` enum 은 설치된 버전에서 실제 멤버를 `dir(Language)` 로 확인 (드리프트 주의)
- [ ] 헤더 metadata 는 vector store 필터 쿼리로 활용 — 검색 recall ↑

## 다음

- `03_semantic_chunker.ipynb` — 임베딩 거리 기반 의미 경계 탐지
- `../03_vectorstores/` — metadata 필터 완전 가이드

## 참고

- 공식: https://docs.langchain.com/oss/python/concepts/text_splitters
- `MarkdownHeaderTextSplitter` API: https://python.langchain.com/api_reference/text_splitters/markdown/langchain_text_splitters.markdown.MarkdownHeaderTextSplitter.html
- `Language` enum 소스: https://github.com/langchain-ai/langchain/blob/master/libs/text-splitters/langchain_text_splitters/base.py